# IEX vs Okta Checker
So sánh trạng thái agent trong Okta snapshot với lịch IEX (dữ liệu đã qua cleaner).

In [1]:
import pandas as pd
from datetime import datetime

In [2]:
# Đọc dữ liệu từ file đã clean (iex_cleaner xuất ra)
iex_df = pd.read_excel('iex-data-extracted.xlsx')
okta_df = pd.read_csv('okta.csv')
iex_df.head()

,IEX Id,Name,Shift,Activity,Start time,End time
0,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Open Time,6:00 AM,6:45 AM
1,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Break,6:45 AM,7:00 AM
2,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Open Time,7:00 AM,12:00 PM
3,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Lunch,12:00 PM,1:00 PM
4,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Open Time,1:00 PM,1:45 PM


In [3]:
# Đổi tên cột Okta cho đồng bộ (nếu cần chỉnh lại tuỳ dataset)
okta_df = okta_df.rename(columns={
    'userName': 'Name',
    'status': 'Activity',
    'duration': 'Duration'
})
okta_df['CheckTime'] = datetime.now()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Manager Email,Business Location,Queue Group / Routing Profile,Forecast Group,CheckTime
0,"Le, Quoc Viet Phuong",07:16:16,ENDOFSHIFT,NaN,quocvietphuong.le@concentrix.com,tiendat.nguyen1@concentrix.com,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125
1,"Le, Thi Hue Anh",00:07:47,BREAK,NaN,thihueanh.le1@concentrix.com,NaN,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125
2,"Nguyen, Bui Thanh Truc",00:07:40,LOGIN,NaN,buithanhtruc.nguyen@concentrix.com,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125
3,"Nguyen, Dinh Thanh Thao",19:26:22,LOGIN,NaN,dinhthanhthao.nguyen@concentrix.com,huynhductran.tran@concentrix.com,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125
4,"Nguyen, Hoang Khoi",01:53:30,LOGIN,NaN,hoangkhoi.nguyen@concentrix.com,thithuytrang.pham1@concentrix.com,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125


In [4]:
# Chuyển Start/End về datetime
iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')
iex_df.head()

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_18228\2049927157.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_18228\2049927157.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')


,IEX Id,Name,Shift,Activity,Start time,End time
0,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Open Time,2025-08-19 06:00:00,2025-08-19 06:45:00
1,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Break,2025-08-19 06:45:00,2025-08-19 07:00:00
2,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Open Time,2025-08-19 07:00:00,2025-08-19 12:00:00
3,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Lunch,2025-08-19 12:00:00,2025-08-19 13:00:00
4,3092836,"Bach, YenNhi",6:00 AM - 3:00 PM,Open Time,2025-08-19 13:00:00,2025-08-19 13:45:00


In [5]:
import re

def clean_name(name: str) -> str:
    if pd.isna(name):
        return name
    # Thay dấu phẩy bằng khoảng trắng
    name = name.replace(",", " ")
    # Thêm khoảng trắng trước chữ in hoa (trừ chữ cái đầu)
    name = re.sub(r'(?<!^)(?=[A-Z])', ' ', name)
    # Chuẩn hoá khoảng trắng thừa
    name = " ".join(name.split())
    return name.strip()

# Chuẩn hoá cho cả IEX và Okta
iex_df["Name"] = iex_df["Name"].astype(str).map(clean_name)
okta_df["Agent Name"] = okta_df["Agent Name"].astype(str).map(clean_name)
#iex_df.head()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Manager Email,Business Location,Queue Group / Routing Profile,Forecast Group,CheckTime
0,Le Quoc Viet Phuong,07:16:16,ENDOFSHIFT,NaN,quocvietphuong.le@concentrix.com,tiendat.nguyen1@concentrix.com,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125
1,Le Thi Hue Anh,00:07:47,BREAK,NaN,thihueanh.le1@concentrix.com,NaN,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125
2,Nguyen Bui Thanh Truc,00:07:40,LOGIN,NaN,buithanhtruc.nguyen@concentrix.com,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125
3,Nguyen Dinh Thanh Thao,19:26:22,LOGIN,NaN,dinhthanhthao.nguyen@concentrix.com,huynhductran.tran@concentrix.com,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125
4,Nguyen Hoang Khoi,01:53:30,LOGIN,NaN,hoangkhoi.nguyen@concentrix.com,thithuytrang.pham1@concentrix.com,Concentrix (Ho Chi Minh City),Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,2025-08-19 05:13:42.709125


In [6]:
# So sánh giữa Okta và IEX
results = []
now = datetime.now()

for _, row in okta_df.iterrows():
    name = row['Agent Name']
    activity_okta = row['State']
    
    # Tìm agent trong IEX
    iex_agent = iex_df[iex_df['Name'] == name]
    iex_now = iex_agent[(iex_agent['Start time'] <= now) & (iex_agent['End time'] >= now)]
    
    if not iex_now.empty:
        activity_iex = iex_now.iloc[0]['Activity']
        start_iex = iex_now.iloc[0]['Start time']
        end_iex = iex_now.iloc[0]['End time']
    else:
        activity_iex = 'N/A'
        start_iex = None
        end_iex = None
    
    results.append({
        'Agent': name,
        'Activity_Okta': activity_okta,
        'Activity_IEX': activity_iex,
        'Start_IEX': start_iex,
        'End_IEX': end_iex,
        'Match': activity_okta.lower() == activity_iex.lower()
    })

result_df = pd.DataFrame(results)
result_df

,Agent,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
0,Le Quoc Viet Phuong,ENDOFSHIFT,N/A,NaT,NaT,False
1,Le Thi Hue Anh,BREAK,Open Time,2025-08-19 04:15:00,2025-08-19 06:00:00,False
2,Nguyen Bui Thanh Truc,LOGIN,N/A,NaT,NaT,False
3,Nguyen Dinh Thanh Thao,LOGIN,N/A,NaT,NaT,False
4,Nguyen Hoang Khoi,LOGIN,N/A,NaT,NaT,False
5,Nguyen Ngoc Hoang Yen,ENDOFSHIFT,N/A,NaT,NaT,False
6,Tran Dan Thanh,TRAINING,Open Time,2025-08-19 02:20:00,2025-08-19 06:40:00,False


In [7]:
# Xuất mismatch ra file Excel
mismatch_df = result_df[(result_df['Match'] == False) & (result_df["Activity_IEX"] != "N/A")]
mismatch_df.to_excel('iex_okta_mismatch.xlsx', index=False)
mismatch_df

,Agent,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
1,Le Thi Hue Anh,BREAK,Open Time,2025-08-19 04:15:00,2025-08-19 06:00:00,False
6,Tran Dan Thanh,TRAINING,Open Time,2025-08-19 02:20:00,2025-08-19 06:40:00,False


In [8]:
# --- Export full comparison with both True/False ---
try:
    out_file = 'iex_okta_comparison.xlsx'
    result_df.to_excel(out_file, index=False)
    print(f'Saved full comparison to: {out_file}')
    result_df
except NameError as e:
    print('result_df is not defined. Please run the comparison cells above first.')
    raise


Saved full comparison to: iex_okta_comparison.xlsx
